## Практика. Введение в прогнозирующие модели

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning) 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
sns.set_style("whitegrid")

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier

>#### Задание 1
Прочитайте файл `logistics`.

In [2]:
df = pd.read_csv('data/logistics.csv')
df

,distance,cargo_weight,cargo_type,departure_hour,day_of_week,season,weather_score,driver_exp,traffic_index,fuel_consumption
0,1172.2,14.6,3,2,5,2,2,19,1.9,26.3
1,686.4,14.4,4,8,0,2,7,23,2.2,29.6
2,1295.0,4.5,1,17,2,1,1,15,6.5,29.5
3,1061.2,1.2,2,10,1,1,8,11,7.4,28.5
4,186.6,6.4,3,4,3,1,1,1,7.8,32.2
...,...,...,...,...,...,...,...,...,...,...
8995,1403.4,17.8,3,0,0,4,10,7,8.9,32.6
8996,1262.4,9.8,1,14,2,3,4,21,1.0,26.8
8997,722.9,9.4,4,1,0,1,8,7,1.9,31.6
8998,1460.8,13.3,3,8,4,1,2,6,9.5,34.1


#### Контекст

Логистическая компания «Быстрый Маршрут» управляет парком из 300 грузовых автомобилей, осуществляющих междугородние перевозки. Компания хочет повысить эффективность маршрутов и снизить операционные расходы. У них есть исторические данные по каждой поездке.


#### Датасет
- distance — протяжённость маршрута (50–1500 км) (float)
- cargo_weight — вес груза (0.5–20 тонн) (float)
- cargo_type — тип груза (1 — хрупкий, 2 — скоропортящийся, 3 — стандартный, 4 — крупногабаритный) (int)
- departure_hour — час отправления (0–23) (int)
- day_of_week — день недели (0 — понедельник, … 6 — воскресенье) (int)
- season — сезон (1 — зима, 2 — весна, 3 — лето, 4 — осень) (int)
- weather_score — балл погоды на маршруте (1–10, где 10 — идеальные условия) (int)
- driver_exp — стаж водителя (1–25 лет) (int)
- traffic_index — средний уровень загруженности трасс по маршруту (1.0–10.0) (float)

#### Таргеты
- fuel_consumption — фактический расход топлива на маршруте (л/100 км) (float)
- delivery_time — фактическое время доставки (часы) (float)
- wear_index — индекс износа автомобиля после рейса (0.0–1.0) (float)


#### Бизнес-задача
- Прогнозируем `fuel_consumption` — чтобы рекомендовать оптимальный маршрут и режим движения, минимизируя затраты на топливо при сохранении времени доставки в допустимых пределах.
- Добавляем `delivery_time`. Модель учится одновременно прогнозировать расход топлива и время доставки. Это позволяет компании выбирать компромисс между скоростью и экономичностью — например, для срочных грузов жертвовать расходом ради скорости, а для стандартных — наоборот. 
- Добавляем третий таргет — `wear_index`, чтобы учитывать долгосрочный износ автопарка и планировать график обслуживания машин.

>#### Задание 2
Обучите регрессионную модель, которая прогнозирует расход топлива. Спрогнозируйте значения таргета для записей из файла `'logistics_new'`.

In [3]:
X = df.drop(columns = ['fuel_consumption'])
y = df['fuel_consumption']

model = LinearRegression()
model.fit(X, y)

LinearRegression()

In [4]:
X_new = pd.read_csv('data/logistics_new.csv')

y_pred = model.predict(X_new)
y_pred[:10]

array([27.14953296, 33.31614488, 31.50979425, 25.16490086, 29.83489411,
       31.59215481, 32.11351633, 33.71253078, 24.4920245 , 31.2865506 ])

>#### Задание 3
Загружите дополнительно еще два предиктора `'delivery_time'` и `'wear_index'` из файла `logistics_extra`. Конкатенируйте их с общим датафреймом. Обучите регрессионную модель, которая прогнозирует разу три таргета: 1) расход топлива, 2) время доставки и 3) индекс износа автомобиля. Спрогнозируйте значения таргетов для записей из файла `'logistics_new'`.

In [5]:
df_extra = pd.read_csv('data/logistics_extra.csv')
df_combine = pd.concat([df, df_extra], axis=1)
df_combine

,distance,cargo_weight,cargo_type,departure_hour,day_of_week,season,weather_score,driver_exp,traffic_index,fuel_consumption,delivery_time,wear_index
0,1172.2,14.6,3,2,5,2,2,19,1.9,26.3,19.72,0.153
1,686.4,14.4,4,8,0,2,7,23,2.2,29.6,12.40,0.103
2,1295.0,4.5,1,17,2,1,1,15,6.5,29.5,24.64,0.147
3,1061.2,1.2,2,10,1,1,8,11,7.4,28.5,19.88,0.088
4,186.6,6.4,3,4,3,1,1,1,7.8,32.2,5.14,0.131
...,...,...,...,...,...,...,...,...,...,...,...,...
8995,1403.4,17.8,3,0,0,4,10,7,8.9,32.6,24.93,0.151
8996,1262.4,9.8,1,14,2,3,4,21,1.0,26.8,22.17,0.106
8997,722.9,9.4,4,1,0,1,8,7,1.9,31.6,13.02,0.114
8998,1460.8,13.3,3,8,4,1,2,6,9.5,34.1,28.41,0.206


In [6]:
X = df_combine.drop(columns = ['fuel_consumption', 'delivery_time', 'wear_index'])
y = df_combine[['fuel_consumption', 'delivery_time', 'wear_index']]

model = LinearRegression()
model.fit(X, y)

LinearRegression()

In [7]:
y_pred = model.predict(X_new)
y_pred[:10]

array([[27.14953296, 24.93045203,  0.11794846],
       [33.31614488,  8.33646146,  0.14177446],
       [31.50979425, 14.69159228,  0.14342499],
       [25.16490086,  1.14223858,  0.05438972],
       [29.83489411,  1.98990069,  0.10778227],
       [31.59215481, 20.7903439 ,  0.14724494],
       [32.11351633,  4.35788071,  0.12454443],
       [33.71253078, 17.08095717,  0.16138079],
       [24.4920245 , 19.12382061,  0.08508831],
       [31.2865506 ,  3.69251846,  0.1110998 ]])

>#### Задание 4
Прочитайте файл `'logistics_missing'`. Он содержит пропуски в признаке `'cargo_weight'`. Импутируйте пропуски при помощи ркгркссионной модели и обучите модель, прогнозитующую расход топлива.

In [8]:
df = pd.read_csv('data/logistics_missing.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   distance          9000 non-null   float64
 1   cargo_weight      8550 non-null   float64
 2   cargo_type        9000 non-null   int64  
 3   departure_hour    9000 non-null   int64  
 4   day_of_week       9000 non-null   int64  
 5   season            9000 non-null   int64  
 6   weather_score     9000 non-null   int64  
 7   driver_exp        9000 non-null   int64  
 8   traffic_index     9000 non-null   float64
 9   fuel_consumption  9000 non-null   float64
dtypes: float64(4), int64(6)
memory usage: 703.3 KB


In [9]:
mask = df['cargo_weight'].isna()
d = df.loc[~mask]

X = d.drop(columns = ['cargo_weight'])
y = d['cargo_weight']

model_history_imputation = LinearRegression()
model_history_imputation.fit(X, y)

LinearRegression()

In [10]:
def imputation(row):
    if row['cargo_weight'] != row['cargo_weight']:
        X_new = row.to_frame().T
        X_new = X_new.drop(columns = ['cargo_weight'])
        imput = round(model_history_imputation.predict(X_new)[0], 2)
        return imput
    else:
        return row['cargo_weight']

In [11]:
df['cargo_weight'] = df.apply(imputation, axis=1)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   distance          9000 non-null   float64
 1   cargo_weight      9000 non-null   float64
 2   cargo_type        9000 non-null   int64  
 3   departure_hour    9000 non-null   int64  
 4   day_of_week       9000 non-null   int64  
 5   season            9000 non-null   int64  
 6   weather_score     9000 non-null   int64  
 7   driver_exp        9000 non-null   int64  
 8   traffic_index     9000 non-null   float64
 9   fuel_consumption  9000 non-null   float64
dtypes: float64(4), int64(6)
memory usage: 703.3 KB


In [12]:
X_new = pd.read_csv('data/logistics_new_missing.csv')
X_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   distance        1000 non-null   float64
 1   cargo_weight    950 non-null    float64
 2   cargo_type      1000 non-null   int64  
 3   departure_hour  1000 non-null   int64  
 4   day_of_week     1000 non-null   int64  
 5   season          1000 non-null   int64  
 6   weather_score   1000 non-null   int64  
 7   driver_exp      1000 non-null   int64  
 8   traffic_index   1000 non-null   float64
dtypes: float64(3), int64(6)
memory usage: 70.4 KB


In [13]:
mask = X_new['cargo_weight'].isna()
d = X_new.loc[~mask]

X = d.drop(columns = ['cargo_weight'])
y = d['cargo_weight']

model_new_imputation = LinearRegression()
model_new_imputation.fit(X, y)

def imputation(row):
    if row['cargo_weight'] != row['cargo_weight']:
        X_new = row.to_frame().T
        X_new = X_new.drop(columns = ['cargo_weight'])
        imput = round(model_new_imputation.predict(X_new)[0], 2)
        return imput
    else:
        return row['cargo_weight']

X_new['cargo_weight'] = X_new.apply(imputation, axis=1)
X_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   distance        1000 non-null   float64
 1   cargo_weight    1000 non-null   float64
 2   cargo_type      1000 non-null   int64  
 3   departure_hour  1000 non-null   int64  
 4   day_of_week     1000 non-null   int64  
 5   season          1000 non-null   int64  
 6   weather_score   1000 non-null   int64  
 7   driver_exp      1000 non-null   int64  
 8   traffic_index   1000 non-null   float64
dtypes: float64(3), int64(6)
memory usage: 70.4 KB


In [14]:
X = df.drop(columns = ['fuel_consumption'])
y = df['fuel_consumption']

model_predict = LinearRegression()
model_predict.fit(X, y)

y_pred = model_predict.predict(X_new)
y_pred[:10]

array([27.07639297, 33.39068712, 31.57235242, 25.11961817, 29.83698866,
       31.64277835, 32.1764004 , 33.73457192, 24.41151038, 31.31460408])

>#### Задание 5
Прочитайте файл `fitnes`.

In [15]:
df = pd.read_csv('data/fitnes.csv')
df

,days_since_last_visit,visits_last_30d,avg_session_len_min,classes_attended,trainer_assigned,promo_used_last_90d,contract_type,days_to_expiry,feedback_score,missed_classes,churn_30d
0,39,2,65.300359,0,0,1,4,54,4.810134,0,0
1,16,2,64.172864,2,0,1,5,31,4.633475,2,0
2,24,4,67.337259,1,1,3,5,35,5.000000,1,0
3,0,8,43.501275,1,0,0,4,49,5.000000,1,0
4,13,4,66.596392,3,0,1,3,36,4.746998,0,1
...,...,...,...,...,...,...,...,...,...,...,...
8995,7,4,45.736654,1,0,0,4,7,4.300161,0,0
8996,26,2,60.452600,2,0,1,3,28,3.334305,6,1
8997,4,13,67.524949,8,1,2,5,51,3.919626,3,0
8998,2,6,57.042357,5,0,0,2,45,4.581053,2,0


#### Контекст
Сеть городских фитнес‑клубов «Энергия Города» хочет повысить удержание клиентов и снизить отток (churn). Сейчас менеджеры реагируют на проблемы постфактум: когда клиент уже решил не продлевать абонемент. Компания хочет внедрить систему раннего предупреждения: выявлять клиентов с высоким риском ухода за 2–4 недели до окончания абонемента, чтобы вовремя предложить персональную акцию, персонального тренера или другую меру удержания. Для этого нужно построить прогнозирующую модель на основе данных о поведении клиентов.

#### Датасет

- days_since_last_visit — количество дней с последнего посещения (0–90) (int)

- visits_last_30d — число посещений за последние 30 дней (0–20) (int)

- avg_session_len_min — средняя длительность тренировки (30–120 минут) (float)

- classes_attended — количество групповых занятий за последние 60 дней (0–15) (int)

- trainer_assigned — закреплён ли персональный тренер (0 — нет, 1 — да) (int)

- promo_used_last_90d — сколько промокодов/скидок использовал клиент за последние 90 дней (0–5) (int)

- contract_type — тип абонемента (1 — разовый, 2 — 1 месяц, 3 — 3 месяца, 4 — 6 месяцев, 5 — 12 месяцев) (int)

- days_to_expiry — дней до окончания действия абонемента (0–60) (int)

- feedback_score — средняя оценка по опросам удовлетворённости (1–5) (float)

- missed_classes — количество пропущенных забронированных групповых занятий за последние 30 дней (0–10) (int)

#### Таргеты

- churn_30d — отток в течение 30 дней после текущей даты (0 — не ушёл, 1 — ушёл) (int)

- low_engagement_30d — низкая вовлечённость в течение 30 дней: менее 2 посещений (0/1) (int)

- upgrade_30d — продление/апгрейд абонемента в течение 30 дней (0 — нет, 1 — да) (int)

#### Бизнес‑задачи

- Прогнозируем churn_30d (0/1) — уйдёт ли клиент в течение 30 дней. Это позволяет отделу удержания заранее связаться с «рисковыми» клиентами, предложить целевые меры и повысить долю удержанных клиентов.

- Добавляем low_engagement_30d (0/1), чтобы параллельно выявлять клиентов, которые формально не уходят, но почти перестают посещать клуб. Раннее вмешательство (персональные рекомендации, напоминания, подбор новых активностей) помогает вернуть вовлечённость до того, как она перерастёт в отток.

- Третий таргет — upgrade_30d (0/1). Модель помогает не только предотвращать потери, но и находить клиентов с высокой вероятностью продления/апгрейда. Для них можно предлагать более выгодные условия заранее, повышая выручку и лояльность.

>#### Задание 6
Обучите классификатор, который прогнозирует уход клиента в течение ближайших 30 дней. Спрогнозируйте значения таргета для записей из файла `'fitnes_new'`.

In [16]:
X = df.drop(columns = ['churn_30d'])
y = df['churn_30d']

model = KNeighborsClassifier()
model.fit(X, y)

KNeighborsClassifier()

In [17]:
X_new = pd.read_csv('data/fitnes_new.csv')

y_pred = model.predict(X_new)
y_pred[:10]

array([1, 0, 0, 0, 0, 1, 0, 0, 0, 0], dtype=int64)

>#### Задание 7
Загружите дополнительно еще два предиктора `'low_engagement_30d'` и `'upgrade_30d'` из файла `fitnes_extra`. Конкатенируйте их с общим датафреймом. Обучите классификатор, который прогнозирует разу три таргета: 1) уход клиента, 2) низкую посещаемость и 3) пролонгацию абонемента. Спрогнозируйте значения таргетов для записей из файла `'fitnes_new'`.

In [18]:
df_extra = pd.read_csv('data/fitnes_extra.csv')
df_combine = pd.concat([df, df_extra], axis=1)
df_combine

,days_since_last_visit,visits_last_30d,avg_session_len_min,classes_attended,trainer_assigned,promo_used_last_90d,contract_type,days_to_expiry,feedback_score,missed_classes,churn_30d,low_engagement_30d,upgrade_30d
0,39,2,65.300359,0,0,1,4,54,4.810134,0,0,0,1
1,16,2,64.172864,2,0,1,5,31,4.633475,2,0,0,1
2,24,4,67.337259,1,1,3,5,35,5.000000,1,0,0,0
3,0,8,43.501275,1,0,0,4,49,5.000000,1,0,0,1
4,13,4,66.596392,3,0,1,3,36,4.746998,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8995,7,4,45.736654,1,0,0,4,7,4.300161,0,0,0,1
8996,26,2,60.452600,2,0,1,3,28,3.334305,6,1,0,0
8997,4,13,67.524949,8,1,2,5,51,3.919626,3,0,0,1
8998,2,6,57.042357,5,0,0,2,45,4.581053,2,0,0,1


In [19]:
X = df_combine.drop(columns = ['churn_30d', 'low_engagement_30d', 'upgrade_30d'])
y = df_combine[['churn_30d', 'low_engagement_30d', 'upgrade_30d']]

model = KNeighborsClassifier()
model.fit(X, y)

KNeighborsClassifier()

In [20]:
y_pred = model.predict(X_new)
y_pred[:10]

array([[1, 0, 0],
       [0, 0, 0],
       [0, 1, 0],
       [0, 1, 0],
       [0, 0, 0],
       [1, 1, 0],
       [0, 0, 1],
       [0, 0, 1],
       [0, 0, 1],
       [0, 0, 0]], dtype=int64)

>#### Задание 4
Прочитайте файл `'fitnes_missing'`. Он содержит пропуски в признаке `'avg_session_len_min'`. Импутируйте пропуски при помощи ркгркссионной модели и обучите модель, прогнозитующую расход топлива.

In [21]:
df = pd.read_csv('data/fitnes_missing.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   days_since_last_visit  9000 non-null   int64  
 1   visits_last_30d        9000 non-null   int64  
 2   avg_session_len_min    8550 non-null   float64
 3   classes_attended       9000 non-null   int64  
 4   trainer_assigned       9000 non-null   int64  
 5   promo_used_last_90d    9000 non-null   int64  
 6   contract_type          9000 non-null   int64  
 7   days_to_expiry         9000 non-null   int64  
 8   feedback_score         9000 non-null   float64
 9   missed_classes         9000 non-null   int64  
 10  churn_30d              9000 non-null   int64  
dtypes: float64(2), int64(9)
memory usage: 773.6 KB


In [22]:
mask = df['avg_session_len_min'].isna()
d = df.loc[~mask]

X = d.drop(columns = ['avg_session_len_min'])
y = d['avg_session_len_min']

model_history_imputation = LinearRegression()
model_history_imputation.fit(X, y)

LinearRegression()

In [23]:
def imputation(row):
    if row['avg_session_len_min'] != row['avg_session_len_min']:
        X_new = row.to_frame().T
        X_new = X_new.drop(columns = ['avg_session_len_min'])
        imput = round(model_history_imputation.predict(X_new)[0], 2)
        return imput
    else:
        return row['avg_session_len_min']

In [24]:
df['avg_session_len_min'] = df.apply(imputation, axis=1)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   days_since_last_visit  9000 non-null   int64  
 1   visits_last_30d        9000 non-null   int64  
 2   avg_session_len_min    9000 non-null   float64
 3   classes_attended       9000 non-null   int64  
 4   trainer_assigned       9000 non-null   int64  
 5   promo_used_last_90d    9000 non-null   int64  
 6   contract_type          9000 non-null   int64  
 7   days_to_expiry         9000 non-null   int64  
 8   feedback_score         9000 non-null   float64
 9   missed_classes         9000 non-null   int64  
 10  churn_30d              9000 non-null   int64  
dtypes: float64(2), int64(9)
memory usage: 773.6 KB


In [25]:
X_new = pd.read_csv('data/fitnes_new_missing.csv')
X_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   days_since_last_visit  1000 non-null   int64  
 1   visits_last_30d        1000 non-null   int64  
 2   avg_session_len_min    950 non-null    float64
 3   classes_attended       1000 non-null   int64  
 4   trainer_assigned       1000 non-null   int64  
 5   promo_used_last_90d    1000 non-null   int64  
 6   contract_type          1000 non-null   int64  
 7   days_to_expiry         1000 non-null   int64  
 8   feedback_score         1000 non-null   float64
 9   missed_classes         1000 non-null   int64  
dtypes: float64(2), int64(8)
memory usage: 78.3 KB


In [26]:
mask = X_new['avg_session_len_min'].isna()
d = X_new.loc[~mask]

X = d.drop(columns = ['avg_session_len_min'])
y = d['avg_session_len_min']

model_new_imputation = LinearRegression()
model_new_imputation.fit(X, y)

def imputation(row):
    if row['avg_session_len_min'] != row['avg_session_len_min']:
        X_new = row.to_frame().T
        X_new = X_new.drop(columns = ['avg_session_len_min'])
        imput = round(model_new_imputation.predict(X_new)[0], 2)
        return imput
    else:
        return row['avg_session_len_min']

X_new['avg_session_len_min'] = X_new.apply(imputation, axis=1)
X_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   days_since_last_visit  1000 non-null   int64  
 1   visits_last_30d        1000 non-null   int64  
 2   avg_session_len_min    1000 non-null   float64
 3   classes_attended       1000 non-null   int64  
 4   trainer_assigned       1000 non-null   int64  
 5   promo_used_last_90d    1000 non-null   int64  
 6   contract_type          1000 non-null   int64  
 7   days_to_expiry         1000 non-null   int64  
 8   feedback_score         1000 non-null   float64
 9   missed_classes         1000 non-null   int64  
dtypes: float64(2), int64(8)
memory usage: 78.3 KB


In [27]:
X = df.drop(columns = ['churn_30d'])
y = df['churn_30d']

model_predict = KNeighborsClassifier()
model_predict.fit(X, y)

y_pred = model_predict.predict(X_new)
y_pred[:10]

array([1, 0, 0, 0, 0, 1, 0, 0, 0, 0], dtype=int64)

## Домашнее задание

#### Замечание
Есть незыблемый принцип: аналитик **обязан** использовать все данные, доступные ему на момент прогноза, и **не имеет права** использовать данные, которые в момент прогноза ему не доступны. 

В приложении к задачам заданий 1 и 2 это означает, что строить импутационные модели сразу с двумя предикторами — **непозволительная роскошь**. Нужно сначала ипутировать пропуски в одном из признаков, а затем, используя уже импутированный стробец в качестве еще одного предиктора — ипмутировать пропуски во втором признаке. 

#### Вопрос
В одном из признаков 5% пропусков, а во втором — 15%. В каком из них нужно проводить импутацию в первую очередь? Как изменится качество прогнозирующей модели, если поменять порядок импутации?

>#### Задание 1
В файлах `logistics_missing_2` и `logistics_new_missing_2` есть пропуски в двух предикторах. Проведите импутацию пропусков, после чего спрогнозируйте расход топлива.

>#### Задание 2
В файлах `fitnes_missing_2` и `fitnes_new_missing_2` есть пропуски в двух предикторах. Проведите импутацию пропусков, после чего спрогнозируйте уход того или иного клиента в течение ближайщих 30 дней.